# 🧠 ICU Clinical Grouping — Single Agent (MedGemma 27B, 4-bit)

The agent reviews the LSTM's top pick and may override it with one of the
other candidates in the top-K. The agent does NOT see LSTM probabilities or
ranking — only which candidate was the top pick, plus the other candidates
as possible overrides.

**Runtime requirement:** Colab GPU with ≥24 GB VRAM (A100 recommended).
MedGemma 27B in 4-bit needs roughly 16–18 GB.

## 1. Setup & Configuration

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q -U transformers accelerate bitsandbytes scikit-learn joblib tqdm
!pip install -U bitsandbytes>=0.46.1

from google.colab import drive, userdata
drive.mount('/content/drive')

print('Setup complete ✓')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 127.6 MB/s eta 0:00:00
Mounted at /content/drive
Setup complete ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import os

# ── File Paths ────────────────────────────────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/702 project'
LSTM_OUTPUT_DIR = f'{BASE_DIR}/lstm_outputs_v2'

CARDS_PATH   = f'{BASE_DIR}/patient_cards_v6_grouped.json'
MODEL_PATH   = f'{LSTM_OUTPUT_DIR}/best_model.pt'
SCALER_PATH  = f'{LSTM_OUTPUT_DIR}/feature_scaler.pkl'
CONFIG_PATH  = f'{LSTM_OUTPUT_DIR}/model_config.json'

AGENT_OUTPUT_DIR = f'{BASE_DIR}/agent_single_medgemma_results'
os.makedirs(AGENT_OUTPUT_DIR, exist_ok=True)

# ── Pilot Configuration ───────────────────────────────────────────────────────
PILOT_N_CASES = 100
TOP_K         = 3
RANDOM_SEED   = 42

# ── LLM Configuration ─────────────────────────────────────────────────────────
# IMPORTANT: use the TEXT-ONLY variant. 'google/medgemma-27b-it' is multimodal
# (Gemma 3 vision+text) and needs AutoModelForImageTextToText — using the
# wrong loader is what caused the empty AttributeError.
MEDGEMMA_MODEL_ID = 'google/medgemma-27b-text-it'

MAX_NEW_TOKENS = 1000
TEMPERATURE    = 0.2
RETRY_ATTEMPTS = 3

FALLBACK_SENTINEL = 'WHITEBOARD JELLYFISH'

print('Configuration:')
print(f'  Cases      : {PILOT_N_CASES}')
print(f'  Top-K      : {TOP_K}')
print(f'  LLM model  : {MEDGEMMA_MODEL_ID}')
print(f'  Output dir : {AGENT_OUTPUT_DIR}')

Configuration:
  Cases      : 100
  Top-K      : 3
  LLM model  : google/medgemma-27b-text-it
  Output dir : /content/drive/MyDrive/702 project/agent_single_medgemma_results


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD MEDGEMMA (text-only, 4-bit quantized)
# ══════════════════════════════════════════════════════════════════════════════
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    print('✓ HF_TOKEN loaded from Colab Secrets')
except Exception:
    raise ValueError('Add HF_TOKEN to Colab Secrets (sidebar → 🔑)')

if not torch.cuda.is_available():
    raise RuntimeError('MedGemma 27B requires a GPU. Switch runtime to GPU.')

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

print(f'\nLoading {MEDGEMMA_MODEL_ID} (this takes a few minutes)...')
llm_tokenizer = AutoTokenizer.from_pretrained(MEDGEMMA_MODEL_ID, token=hf_token)
llm_model = AutoModelForCausalLM.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=quantization_config,
    device_map='auto',
    token=hf_token,
)
llm_model.eval()

if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token_id = llm_tokenizer.eos_token_id

print(f'✓ MedGemma loaded ({sum(p.numel() for p in llm_model.parameters()):,} params)')

✓ HF_TOKEN loaded from Colab Secrets
GPU : NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB

Loading google/medgemma-27b-text-it (this takes a few minutes)...


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

✓ MedGemma loaded (14,209,821,440 params)


## 2. Load Data & LSTM Model

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD PATIENT CARDS & SCALER
# ══════════════════════════════════════════════════════════════════════════════
import json
import numpy as np
import random
import torch.nn as nn
import joblib
from collections import defaultdict, Counter

print('Loading patient cards...')
with open(CARDS_PATH) as f:
    data = json.load(f)

all_cards      = data['patients']
label_map      = data['label_map']
feature_names  = data['feature_names']
n_features     = data['n_features']
n_timesteps    = data['n_timesteps']
num_classes    = len(label_map)
inv_label_map  = {v: k for k, v in label_map.items()}

val_idx  = list(range(0, n_features, 2))
mask_idx = list(range(1, n_features, 2))

print(f'  Patients: {len(all_cards):,} | Classes: {num_classes}')

with open(CONFIG_PATH) as f:
    config = json.load(f)

scaler = joblib.load(SCALER_PATH)
print('✓ Data + scaler loaded')

Loading patient cards...
  Patients: 28,467 | Classes: 21
✓ Data + scaler loaded


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD LSTM MODEL  (kept as `lstm_model` to avoid collision with `llm_model`)
# ══════════════════════════════════════════════════════════════════════════════
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class LSTMWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 dropout=0.3, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=True
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_attention=False):
        lstm_out, _ = self.lstm(x)
        attn_out, attn_weights = self.attention(lstm_out, lstm_out, lstm_out, need_weights=True)
        attn_out = self.layer_norm(lstm_out + attn_out)
        pooled = attn_out.mean(dim=1)
        out = self.dropout(pooled)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        logits = self.fc2(out)
        return (logits, attn_weights) if return_attention else logits

lstm_model = LSTMWithAttention(
    input_dim=config['input_dim'], hidden_dim=config['hidden_dim'],
    num_layers=config['num_layers'], num_classes=config['num_classes'],
    dropout=config['dropout'], num_heads=config['num_heads']
).to(device)

lstm_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
lstm_model.eval()
print(f'✓ LSTM loaded ({sum(p.numel() for p in lstm_model.parameters()):,} params)')

✓ LSTM loaded (3,572,245 params)


In [ ]:
# # ══════════════════════════════════════════════════════════════════════════════
# # CREATE TEST SPLIT & PILOT SAMPLE
# # ══════════════════════════════════════════════════════════════════════════════
# TRAIN_RATIO, VAL_RATIO = 0.70, 0.15

# random.seed(42)
# np.random.seed(42)

# patient_to_cards = defaultdict(list)
# for i, card in enumerate(all_cards):
#     patient_to_cards[card['patient_id']].append(i)

# patient_ids = list(patient_to_cards.keys())
# random.shuffle(patient_ids)

# n_train = int(len(patient_ids) * TRAIN_RATIO)
# n_val   = int(len(patient_ids) * VAL_RATIO)
# test_pids  = set(patient_ids[n_train + n_val:])
# test_cards = [all_cards[i] for pid in test_pids for i in patient_to_cards[pid]]

# random.seed(RANDOM_SEED)
# label_to_cards = defaultdict(list)
# for card in test_cards:
#     label_to_cards[card['label']].append(card)

# label_counts = Counter(c['label'] for c in test_cards)
# total = sum(label_counts.values())

# pilot_cards = []
# for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
#     n_sample = max(1, int(PILOT_N_CASES * count / total))
#     n_sample = min(n_sample, len(label_to_cards[label]))
#     pilot_cards.extend(random.sample(label_to_cards[label], n_sample))

# random.shuffle(pilot_cards)
# pilot_cards = pilot_cards[:PILOT_N_CASES]

# print(f'Test set   : {len(test_cards):,} stays')
# print(f'Pilot sample: {len(pilot_cards)} cases')

Test set   : 4,336 stays
Pilot sample: 96 cases


In [ ]:
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15

random.seed(42)
np.random.seed(42)

patient_to_cards = defaultdict(list)
for i, card in enumerate(all_cards):
    patient_to_cards[card['patient_id']].append(i)

patient_ids = list(patient_to_cards.keys())
random.shuffle(patient_ids)

n_train = int(len(patient_ids) * TRAIN_RATIO)
n_val   = int(len(patient_ids) * VAL_RATIO)
test_pids_ordered = patient_ids[n_train + n_val:]   # already unique, already shuffled deterministically
test_cards = [all_cards[i]
              for pid in test_pids_ordered
              for i in patient_to_cards[pid]]

random.seed(RANDOM_SEED)

# Index test cards so we can de-dup deterministically across stratification + top-up
indexed_test = list(enumerate(test_cards))
label_to_indexed = defaultdict(list)
for ix, card in indexed_test:
    label_to_indexed[card['label']].append((ix, card))

label_counts = Counter(c['label'] for c in test_cards)
total = sum(label_counts.values())

pilot_indices = set()
pilot_cards   = []

# 2a. Proportional allocation per class (uses int(), so under-allocates)
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    n_sample = max(1, int(PILOT_N_CASES * count / total))
    n_sample = min(n_sample, len(label_to_indexed[label]))
    for ix, c in random.sample(label_to_indexed[label], n_sample):
        pilot_indices.add(ix)
        pilot_cards.append(c)

# 2b. Top up to PILOT_N_CASES from any unused test cards (rounding-loss correction)
shortfall = PILOT_N_CASES - len(pilot_cards)
if shortfall > 0:
    remaining = [(ix, c) for ix, c in indexed_test if ix not in pilot_indices]
    n_extra = min(shortfall, len(remaining))
    if n_extra > 0:
        for ix, c in random.sample(remaining, n_extra):
            pilot_indices.add(ix)
            pilot_cards.append(c)

print(f'Test set   : {len(test_cards):,} stays')
print(f'Pilot sample: {len(pilot_cards)} cases')


Test set   : 4,336 stays
Pilot sample: 100 cases


In [ ]:
import hashlib
pilot_signature = hashlib.md5(
    ','.join(str(c['patient_id']) + ':' + str(c.get('stay_id', '')) for c in pilot_cards).encode()
).hexdigest()
print(f'pilot signature: {pilot_signature}')
print(f'first 5 patient_ids: {[c["patient_id"] for c in pilot_cards[:5]]}')
print(f'last 5 patient_ids:  {[c["patient_id"] for c in pilot_cards[-5:]]}')

pilot signature: 782d14bb87559198308d5f480e180477
first 5 patient_ids: ['19049931', '11334897', '13169821', '19405247', '18344437']
last 5 patient_ids:  ['13706429', '10884861', '10904094', '19646802', '18765416']


## 3. Define Single Agent System

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING & LSTM PREDICTION
# ══════════════════════════════════════════════════════════════════════════════

def normalize_time_series(card):
    """Apply scaler to patient time series."""
    X = np.array(card['time_series'], dtype=np.float32)
    X_vals = X[:, val_idx]
    X_vals_scaled = scaler.transform(X_vals)
    X[:, val_idx] = X_vals_scaled.astype(np.float32)
    return X


def get_lstm_predictions(card, top_k=3):
    """Get LSTM top-K predictions."""
    X = normalize_time_series(card)
    X_tensor = torch.from_numpy(X).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = lstm_model(X_tensor)
        probs = torch.softmax(logits, dim=-1).squeeze(0)

    top_k_idx    = probs.argsort(descending=True)[:top_k]
    top_k_groups = [inv_label_map[i.item()] for i in top_k_idx]
    top_k_probs  = [probs[i].item() for i in top_k_idx]

    return {
        'top_k_codes'    : top_k_groups,
        'top_k_probs'    : top_k_probs,
        'lstm_prediction': top_k_groups[0],
        'lstm_confidence': top_k_probs[0],
    }

test_pred = get_lstm_predictions(pilot_cards[0], TOP_K)
print(f'✓ LSTM test: {test_pred["lstm_prediction"]} ({test_pred["lstm_confidence"]:.1%})')

✓ LSTM test: SEPSIS (33.3%)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FORMATTERS
# ══════════════════════════════════════════════════════════════════════════════

def format_clinical_summary(summary: dict) -> str:
    """Format clinical summary for prompt."""
    if not summary:
        return "Clinical summary not available"

    lines = []

    if 'vitals' in summary:
        lines.append("VITAL SIGNS (Last 12 Hours):")
        for name, stats in summary['vitals'].items():
            if isinstance(stats, dict) and 'mean' in stats:
                lines.append(
                    f"  {name.replace('_', ' ').title()}: "
                    f"mean={stats['mean']:.1f}, range=[{stats['min']:.1f}–{stats['max']:.1f}], "
                    f"last={stats['last']:.1f} ({stats.get('trend', 'N/A')})"
                )

    if 'labs' in summary:
        lines.append("\nLABORATORY VALUES:")
        for name, stats in summary['labs'].items():
            if isinstance(stats, dict) and 'mean' in stats:
                lines.append(
                    f"  {name.replace('_', ' ').title()}: "
                    f"mean={stats['mean']:.2f}, last={stats['last']:.2f} ({stats.get('trend', 'N/A')})"
                )

    if 'interventions' in summary:
        iv = summary['interventions']
        lines.append("\nINTERVENTIONS:")
        if iv.get('active_drugs'):
            lines.append(f"  Medications: {', '.join(iv['active_drugs'])}")
        if iv.get('active_procedures'):
            lines.append(f"  Procedures: {', '.join(iv['active_procedures'])}")

    return '\n'.join(lines) if lines else "Clinical summary not available"

print('✓ Formatters defined')

✓ Formatters defined


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD AGENT-SAFE PATIENT CARD
# ══════════════════════════════════════════════════════════════════════════════
# Strip any field that reveals (or temporally leaks) the diagnosis before the
# card is handed to the LLM. Ground truth stays on the ORIGINAL card for
# downstream validation.

LEAKY_CARD_FIELDS = {
    'icd_code',         # raw ICD-10 code
    'clinical_group',   # text answer
    'label',            # integer index into label_map
    'procedures',       # whole-admission ICD-PCS codes (temporal leak)
    'drugs',            # whole-admission prescriptions (temporal leak)
    'conditions',       # may contain dx history
}


def build_clinical_summary_for_agent(card: dict) -> dict:
    """Return a shallow copy of the card with diagnosis-revealing fields removed."""
    safe = {k: v for k, v in card.items() if k not in LEAKY_CARD_FIELDS}

    # Defensive scrub of clinical_summary keys whose names hint at diagnosis.
    cs = safe.get('clinical_summary')
    if isinstance(cs, dict):
        diag_hint_keys = [
            k for k in cs.keys()
            if any(tok in k.lower()
                   for tok in ('diag', 'icd', 'group', 'label', 'condition',
                               'problem', 'impression'))
        ]
        if diag_hint_keys:
            safe['clinical_summary'] = {
                k: v for k, v in cs.items() if k not in diag_hint_keys
            }

    return safe


_probe = build_clinical_summary_for_agent(pilot_cards[0])
_leaks_present = LEAKY_CARD_FIELDS & set(_probe.keys())
assert not _leaks_present, f'Leak check failed: {_leaks_present} still in stripped card'
print('✓ build_clinical_summary_for_agent() defined')
print(f'  Original card keys   : {sorted(pilot_cards[0].keys())}')
print(f'  Agent-safe card keys : {sorted(_probe.keys())}')
print(f'  Stripped             : {sorted(set(pilot_cards[0].keys()) - set(_probe.keys()))}')

✓ build_clinical_summary_for_agent() defined
  Original card keys   : ['clinical_group', 'clinical_summary', 'conditions', 'drugs', 'feature_names', 'icd_code', 'label', 'patient_id', 'procedures', 'stay_id', 'time_series', 'visit_id']
  Agent-safe card keys : ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']
  Stripped             : ['clinical_group', 'conditions', 'drugs', 'icd_code', 'label', 'procedures']


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSPECT AGENT-SAFE CARDS (pre-flight audit)
# ══════════════════════════════════════════════════════════════════════════════
# Nothing here is shown to the LLM — this is for you to sanity-check what the
# agent will and won't see.
import pprint

N_INSPECT = 5
pp = pprint.PrettyPrinter(width=100, depth=4, sort_dicts=False)

for inspect_i, inspect_card in enumerate(pilot_cards[:N_INSPECT]):
    print('\n' + '█' * 78)
    print(f'█ AGENT-SAFE CARD PREVIEW  [{inspect_i + 1}/{N_INSPECT}]')
    print('█' * 78)

    _true = inv_label_map[inspect_card['label']]
    print(f'\n➤ GROUND TRUTH (not shown to agent):  {_true}')
    print(f'  (integer label {inspect_card["label"]}, patient {inspect_card["patient_id"]})')

    safe = build_clinical_summary_for_agent(inspect_card)
    print('\n── 1. AGENT-SAFE CARD (top-level keys) ──')
    print(f'  Keys     : {sorted(safe.keys())}')
    print(f'  Stripped : {sorted(set(inspect_card.keys()) - set(safe.keys()))}')

    print('\n── 2. FULL clinical_summary DICT ──')
    pp.pprint(safe.get('clinical_summary', {}))

    print('\n── 3. FORMATTED CLINICAL SUMMARY (exact text in prompt) ──')
    print(format_clinical_summary(safe.get('clinical_summary', {})))

    _lstm = get_lstm_predictions(safe, TOP_K)
    print('\n── 4. LSTM TOP-K (agent sees top pick + other candidates, no probs) ──')
    print(f'  LSTM top-{TOP_K} with probs (for your reference only):')
    for _rank, (_c, _p) in enumerate(zip(_lstm['top_k_codes'], _lstm['top_k_probs'])):
        _mark = '✓' if _c == _true else ' '
        print(f'    {_rank+1}. {_c:<25} {_p:>6.1%} {_mark}')
    print(f'  What the agent will see:')
    print(f'    LSTM TOP PICK    : {_lstm["top_k_codes"][0]}')
    print(f'    Other candidates :')
    for _c in _lstm['top_k_codes'][1:]:
        print(f'      - {_c}')

print('\n' + '█' * 78)
print(f'█ End of agent-safe card preview ({N_INSPECT} cards)')
print('█' * 78)


██████████████████████████████████████████████████████████████████████████████
█ AGENT-SAFE CARD PREVIEW  [1/5]
██████████████████████████████████████████████████████████████████████████████

➤ GROUND TRUTH (not shown to agent):  SEPSIS
  (integer label 12, patient 19049931)

── 1. AGENT-SAFE CARD (top-level keys) ──
  Keys     : ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']
  Stripped : ['clinical_group', 'conditions', 'drugs', 'icd_code', 'label', 'procedures']

── 2. FULL clinical_summary DICT ──
{'vitals': {'heart_rate': {'mean': 98.33,
                           'min': 92.0,
                           'max': 108.0,
                           'last': 101.0,
                           'first': 100.0,
                           'trend': 'stable',
                           'n_observations': 9},
            'respiratory_rate': {'mean': 32.94,
                                 'min': 21.0,
                                 'max': 41.5,
       

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SINGLE AGENT PROMPT — BLINDED to LSTM ranking
# ══════════════════════════════════════════════════════════════════════════════

SINGLE_AGENT_PROMPT = """
You are an expert ICU physician. You are given patient data and {top_k} possible diagnoses to consider. Your task is to determine which diagnosis best fits the clinical picture.

## PATIENT DATA (Last 12 Hours in ICU)
{clinical_summary}

## CANDIDATE DIAGNOSES
Consider these {top_k} possibilities (listed in no particular order):
{candidates_list}


## YOUR TASK
You are working with INCOMPLETE ICU data. Not all clinically relevant labs will be present. Your goal is to find the diagnosis most consistent with available evidence, explicitly accounting for what is and is not measured. A diagnosis should not be dismissed solely because its defining marker was not collected.


Respond with ONLY valid JSON:
{{
  "chosen_diagnosis": "DIAGNOSIS_NAME",
  "reasoning": "2-3 sentence explanation of why this diagnosis fits best",
  "supporting_evidence": ["key finding 1", "key finding 2", "key finding 3"],
  "ruled_out": [
    {{"diagnosis": "OTHER_DIAGNOSIS_1", "reason": "why ruled out"}},
    {{"diagnosis": "OTHER_DIAGNOSIS_2", "reason": "why ruled out"}}
  ]
}}

Your chosen_diagnosis MUST be one of the {top_k} candidates listed above.
"""

print('✓ Blinded single agent prompt defined (agent does NOT see LSTM ranking)')

✓ Blinded single agent prompt defined (agent does NOT see LSTM ranking)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LLM WRAPPER (local MedGemma) — robust error handling + full tracebacks
# ══════════════════════════════════════════════════════════════════════════════
import re
import time
import traceback


def _is_oom(exc: BaseException) -> bool:
    """Detect GPU OOM across PyTorch versions.

    torch.cuda.OutOfMemoryError was only added in PyTorch 2.1; on older
    installs we fall back to sniffing the RuntimeError message.
    """
    oom_cls = getattr(torch.cuda, 'OutOfMemoryError', None)
    if oom_cls is not None and isinstance(exc, oom_cls):
        return True
    return isinstance(exc, RuntimeError) and 'out of memory' in str(exc).lower()


def call_llm(prompt: str) -> dict:
    """Call MedGemma 27B (local, 4-bit) and parse JSON from the response."""

    for attempt in range(RETRY_ATTEMPTS):
        try:
            # 1. Render chat template to a plain string, then tokenize explicitly.
            #    This avoids transformers-version differences in what
            #    apply_chat_template(return_tensors='pt') returns (tensor vs
            #    BatchEncoding) that caused the empty AttributeError.
            chat_text = llm_tokenizer.apply_chat_template(
                [{'role': 'user', 'content': prompt}],
                add_generation_prompt=True,
                tokenize=False,
            )
            inputs = llm_tokenizer(
                chat_text,
                return_tensors='pt',
            ).to(llm_model.device)

            # 2. Generate
            with torch.no_grad():
                outputs = llm_model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE,
                    pad_token_id=llm_tokenizer.pad_token_id,
                )

            # 3. Decode only new tokens
            prompt_len = inputs['input_ids'].shape[-1]
            new_tokens = outputs[0][prompt_len:]
            full_text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)

            if not full_text.strip():
                raise ValueError('Empty response from model')

            # 4. Extract JSON object from response
            json_match = re.search(r'\{[\s\S]*\}', full_text.strip())
            if not json_match:
                raise ValueError(f'No JSON found. First 200 chars: {full_text[:200]!r}')
            return json.loads(json_match.group(0))

        except Exception as e:
            if _is_oom(e):
                print('    ⚠ GPU OOM: clearing cache and retrying...')
                torch.cuda.empty_cache()
                time.sleep(5)
                continue

            # Show the full traceback — this is what we needed to debug the
            # original empty-AttributeError. Truncated str(e) hid everything.
            print(f'    ⚠ {type(e).__name__} on attempt {attempt+1}/{RETRY_ATTEMPTS}:')
            traceback.print_exc()
            if attempt < RETRY_ATTEMPTS - 1:
                time.sleep(2)

    return {'error': 'Failed after retries', 'chosen_diagnosis': None}

print('✓ MedGemma wrapper ready')

✓ MedGemma wrapper ready


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SINGLE AGENT DIAGNOSIS FUNCTION — BLINDED mode
# ══════════════════════════════════════════════════════════════════════════════
# The agent sees all K candidates in a SHUFFLED order, with no indication of
# which was the LSTM's top pick. 'accept' / 'override' labels are computed
# post-hoc by comparing the agent's choice to the LSTM top-1 — the agent itself
# is never told what the LSTM ranked first.


def diagnose_single_agent(patient_card: dict, verbose: bool = False) -> dict:
    """Run single-agent diagnosis on one patient, blinded to LSTM ranking."""

    def _log(title, body=None):
        if not verbose:
            return
        print(f'\n── {title} ──')
        if body is not None:
            print(body)

    # Build agent-safe view of the card. Everything the LLM sees comes from here.
    clinical_summary_for_agent = build_clinical_summary_for_agent(patient_card)
    _log('STEP A: agent-safe card (top-level keys)',
         f'  keys: {sorted(clinical_summary_for_agent.keys())}')

    # 1. LSTM top-K (uses stripped card — only needs time_series)
    lstm_output = get_lstm_predictions(clinical_summary_for_agent, TOP_K)
    if verbose:
        print('\n── STEP B: LSTM top-K (probs shown for reference — NOT shown to agent) ──')
        for _rank, (_c, _p) in enumerate(zip(lstm_output['top_k_codes'], lstm_output['top_k_probs'])):
            print(f'  {_rank+1}. {_c:<25} {_p:>6.1%}')

    # 2. Record LSTM top-1 for post-hoc agreement analysis (NOT shown to agent)
    lstm_top_pick = lstm_output['top_k_codes'][0]

    # 3. Shuffle the candidates. Seed per-patient so the order is reproducible
    #    across runs but independent of LSTM rank position.
    patient_id_for_seed = patient_card.get('patient_id') or patient_card.get('stay_id') or 0
    shuffle_rng = random.Random(hash(('shuffle', patient_id_for_seed, RANDOM_SEED)))
    shuffled_candidates = list(lstm_output['top_k_codes'])
    shuffle_rng.shuffle(shuffled_candidates)

    _log('STEP C: shuffled candidate order the agent will see (blinded)',
         '  ' + ' | '.join(shuffled_candidates))

    candidates_list = '\n'.join([f'- {code}' for code in shuffled_candidates])

    # 4. Build the prompt
    formatted_summary = format_clinical_summary(
        clinical_summary_for_agent.get('clinical_summary', {})
    )
    _log('STEP D: formatted clinical summary', formatted_summary)

    prompt = SINGLE_AGENT_PROMPT.format(
        top_k=TOP_K,
        clinical_summary=formatted_summary,
        candidates_list=candidates_list,
    )
    _log('STEP E: FULL PROMPT SENT TO LLM', prompt)

    # 5. Call LLM
    agent_output = call_llm(prompt)
    if verbose:
        print('\n── STEP F: RAW LLM RESPONSE (parsed JSON) ──')
        try:
            print(json.dumps(agent_output, indent=2, ensure_ascii=False))
        except (TypeError, ValueError):
            print(repr(agent_output))

    # 6. Extract prediction; normalize against the candidate set
    raw_pred = agent_output.get('chosen_diagnosis') or ''
    norm     = raw_pred.strip().strip('"\'.,').upper()
    lookup   = {c.strip().upper(): c for c in lstm_output['top_k_codes']}
    agent_pred = lookup.get(norm)

    if agent_pred is None:
        if verbose:
            print(f'\n  ⚠ chosen_diagnosis {raw_pred!r} not in top-K candidates; '
                  f'falling back to sentinel: {FALLBACK_SENTINEL}')
        agent_pred = FALLBACK_SENTINEL

    # 7. Compute post-hoc agreement label (agent itself never reported a decision)
    decision = 'accept' if agent_pred == lstm_top_pick else 'override'

    print(f'{agent_pred}  ({decision})')

    if verbose:
        print('\n── STEP G: AGENT REASONING (parsed fields) ──')
        print(f'  decision (post-hoc): {decision}   [agent was blinded; computed from chosen_diagnosis vs LSTM top-1]')
        print(f'  chosen_diagnosis   : {agent_pred}')
        _reasoning = agent_output.get('reasoning')
        if _reasoning:
            print('  reasoning        :')
            for line in str(_reasoning).splitlines() or [str(_reasoning)]:
                print(f'      {line}')
        _support = agent_output.get('supporting_evidence') or []
        if _support:
            print('  supporting_evidence:')
            for item in _support:
                print(f'      • {item}')
        _ruled = agent_output.get('ruled_out') or []
        if _ruled:
            print('  ruled_out:')
            for item in _ruled:
                if isinstance(item, dict):
                    print(f'      • {item.get("diagnosis", "?")}: {item.get("reason", "")}')
                else:
                    print(f'      • {item}')

    # 8. Ground truth from the ORIGINAL card (never sent to LLM)
    true_label = inv_label_map[patient_card['label']]

    return {
        'patient_id'            : patient_card.get('patient_id'),
        'stay_id'               : patient_card.get('stay_id'),
        'true_label'            : true_label,
        'lstm_output'           : lstm_output,
        'lstm_top_pick'         : lstm_top_pick,
        'candidates_shown_order': shuffled_candidates,  # exact order agent saw
        'agent_output'          : agent_output,
        'agent_prediction'      : agent_pred,
        'agent_decision'        : decision,             # post-hoc label
        'agent_disagrees'       : agent_pred != lstm_top_pick,
        'lstm_correct'          : lstm_top_pick == true_label,
        'agent_correct'         : agent_pred == true_label,
    }

print('✓ Blinded single agent system ready')

✓ Blinded single agent system ready


## 4. Run Pilot

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RUN PILOT  (verbose: full per-case logging)
# ══════════════════════════════════════════════════════════════════════════════
from tqdm.notebook import tqdm
import datetime

print(f'Starting single-agent pilot: {len(pilot_cards)} cases')
print(f'LLM calls: {len(pilot_cards)} (1 per case)')
print(f'Model   : {MEDGEMMA_MODEL_ID}')
print(f'Verbose : True  — every step of every case will be printed in full.')
print('=' * 78)

results = []
start_time = datetime.datetime.now()

for i, card in enumerate(tqdm(pilot_cards, desc='Processing')):
    true_label = inv_label_map[card['label']]

    print('\n' + '═' * 78)
    print(f'CASE [{i+1}/{len(pilot_cards)}]  Patient {card["patient_id"]}  |  True: {true_label}')
    print('═' * 78)

    try:
        result = diagnose_single_agent(card, verbose=True)
        results.append(result)

        lstm_mark    = '✓' if result['lstm_correct']  else '✗'
        agent_mark   = '✓' if result['agent_correct'] else '✗'
        decision_tag = f'  ({result["agent_decision"].upper()})'
        print('\n── OUTCOME ──')
        print(f'  LSTM top pick : {result["lstm_top_pick"]} {lstm_mark}')
        print(f'  Agent pick    : {result["agent_prediction"]} {agent_mark}{decision_tag}')

    except Exception as e:
        print(f'\n  ERROR: {type(e).__name__}: {e}')
        results.append({'patient_id': card.get('patient_id'), 'error': str(e)})

elapsed = datetime.datetime.now() - start_time
print('\n' + '=' * 78)
print(f'Pilot complete — elapsed: {elapsed}')
print(f'Valid: {sum(1 for r in results if "error" not in r)}/{len(results)}')

Starting single-agent pilot: 100 cases
LLM calls: 100 (1 per case)
Model   : google/medgemma-27b-text-it
Verbose : True  — every step of every case will be printed in full.


Processing:   0%|          | 0/100 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.

## YOUR TASK
You are working with INCOMPLETE ICU data. Not all clinically relevant labs will be present. Your goal is to find the diagnosis most consistent with available evidence, explicitly accounting for what is and is not measured. A diagnosis should not be dismissed solely because its defining marker was not collected.


Respond with ONLY valid JSON:
{
  "chosen_diagnosis": "DIAGNOSIS_NAME",
  "reasoning": "2-3 sentence explanation of why this diagnosis fits best",
  "supporting_evidence": ["key finding 1", "key finding 2", "key finding 3"],
  "ruled_out": [
    {"diagnosis": "OTHER_DIAGNOSIS_1", "reason": "why ruled out"},
    {"diagnosis": "OTHER_DIAGNOSIS_2", "reason": "why ruled out"}
  ]
}

Your chosen_diagnosis MUST be one of the 3 candidates listed above.


── STEP F: RAW LLM RESPONSE (parsed JSON) ──
{
  "chosen_diagnosis": "SURGICAL_COMPLICATION",
  "reasoning": "The patient exhibits worsening hemodynamics (decreasing bl

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE METRICS
# ══════════════════════════════════════════════════════════════════════════════

valid_results = [r for r in results if 'error' not in r]
print(f'Valid results: {len(valid_results)}/{len(results)}')

disagree_cases = []
if valid_results:
    n = len(valid_results)

    lstm_correct  = sum(r['lstm_correct']  for r in valid_results)
    agent_correct = sum(r['agent_correct'] for r in valid_results)

    both_correct   = sum(r['lstm_correct']     and r['agent_correct']     for r in valid_results)
    agent_improved = sum(not r['lstm_correct'] and r['agent_correct']     for r in valid_results)
    agent_hurt     = sum(r['lstm_correct']     and not r['agent_correct'] for r in valid_results)
    both_wrong     = sum(not r['lstm_correct'] and not r['agent_correct'] for r in valid_results)

    disagreements = sum(r['agent_disagrees'] for r in valid_results)
    true_in_topk  = sum(r['true_label'] in r['lstm_output']['top_k_codes'] for r in valid_results)

    print(f'\n{"═" * 60}')
    print(f'SINGLE AGENT RESULTS ({n} cases, top-{TOP_K})')
    print(f'{"═" * 60}')
    print(f'\nAccuracy:')
    print(f'  LSTM Top-1  : {lstm_correct}/{n} ({lstm_correct/n*100:.1f}%)')
    print(f'  Agent       : {agent_correct}/{n} ({agent_correct/n*100:.1f}%)')
    print(f'  Improvement : {(agent_correct - lstm_correct)/n*100:+.1f}%')
    print(f'\nTrue in top-{TOP_K}: {true_in_topk}/{n} ({true_in_topk/n*100:.1f}%)')
    print(f'Agent disagreed: {disagreements}/{n} ({disagreements/n*100:.1f}%)')
    print(f'\nCase Breakdown:')
    print(f'  Both correct    : {both_correct} ({both_correct/n*100:.1f}%)')
    print(f'  Agent improved  : {agent_improved} ({agent_improved/n*100:.1f}%) 🎉')
    print(f'  Agent hurt      : {agent_hurt} ({agent_hurt/n*100:.1f}%) ⚠️')
    print(f'  Both wrong      : {both_wrong} ({both_wrong/n*100:.1f}%)')
    print(f'\nNet: {agent_improved - agent_hurt:+d} cases')

    disagree_cases = [r for r in valid_results if r['agent_disagrees']]

Valid results: 100/100

════════════════════════════════════════════════════════════
SINGLE AGENT RESULTS (100 cases, top-3)
════════════════════════════════════════════════════════════

Accuracy:
  LSTM Top-1  : 41/100 (41.0%)
  Agent       : 32/100 (32.0%)
  Improvement : -9.0%

True in top-3: 78/100 (78.0%)
Agent disagreed: 47/100 (47.0%)

Case Breakdown:
  Both correct    : 23 (23.0%)
  Agent improved  : 9 (9.0%) 🎉
  Agent hurt      : 18 (18.0%) ⚠️
  Both wrong      : 50 (50.0%)

Net: -9 cases


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSPECT A CASE
# ══════════════════════════════════════════════════════════════════════════════

def inspect_case(result):
    print('═' * 70)
    print(f"PATIENT: {result['patient_id']}")
    print('═' * 70)
    print(f"\n🎯 TRUE: {result['true_label']}")

    print(f"\n📊 LSTM TOP-{TOP_K} (probs not shown to agent):")
    for i, (code, prob) in enumerate(zip(
        result['lstm_output']['top_k_codes'],
        result['lstm_output']['top_k_probs']
    )):
        mark = '✓' if code == result['true_label'] else ' '
        tag  = '  ← LSTM top pick' if i == 0 else ''
        print(f"   {i+1}. {code:<25} {prob:>6.1%} {mark}{tag}")

    print(f"\n   What the agent saw (no probs):")
    print(f"     LSTM top pick   : {result.get('lstm_top_pick')}")
    for c in result.get('other_candidates_shown', []):
        print(f"     other candidate : {c}")

    print(f"\n🤖 AGENT DECISION:")
    print(f"   Decision: {result.get('agent_decision', '?').upper()}")
    print(f"   Chose   : {result['agent_prediction']}")

    ao = result['agent_output']
    if 'lstm_top_pick_assessment' in ao:
        print(f"\n   LSTM top-pick assessment: {ao['lstm_top_pick_assessment']}")
    if 'reasoning' in ao:
        print(f"\n   Reasoning: {ao['reasoning']}")
    if 'supporting_evidence' in ao:
        print(f"\n   Supporting evidence:")
        for ev in ao['supporting_evidence'][:3]:
            print(f"     • {ev}")
    if 'ruled_out' in ao:
        print(f"\n   Ruled out:")
        for ro in ao['ruled_out']:
            print(f"     • {ro.get('diagnosis', '?')}: {ro.get('reason', 'N/A')}")

    print(f"\n{'─' * 70}")
    if result['lstm_correct'] and result['agent_correct']:
        print('✅ BOTH CORRECT')
    elif not result['lstm_correct'] and result['agent_correct']:
        print('🎉 AGENT IMPROVED (override was right)' if result['agent_decision'] == 'override'
              else '🎉 AGENT CORRECT (accepted correct LSTM pick)')
    elif result['lstm_correct'] and not result['agent_correct']:
        print('⚠️ AGENT HURT (overrode a correct LSTM pick)')
    else:
        print('❌ BOTH WRONG')

if valid_results:
    inspect_case(valid_results[0])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DISAGREEMENTS WHERE TRUE IS IN LSTM TOP-K
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '═' * 60)
print('DISAGREEMENTS WHERE TRUE IS IN LSTM TOP-K')
print('═' * 60)

true_in_topk_disagreements = [
    r for r in disagree_cases
    if r['true_label'] in r['lstm_output']['top_k_codes']
]

if true_in_topk_disagreements:
    for r in true_in_topk_disagreements:
        print(f'\n  Patient {r["patient_id"]}')
        print(f'    True: {r["true_label"]}')
        print(f'    LSTM Top-{TOP_K}: {r["lstm_output"]["top_k_codes"]}')
        print(f'    Agent: {r["agent_prediction"]}')
    for result_item in true_in_topk_disagreements:
        inspect_case(result_item)
else:
    print('  No cases where agent disagreed and true label was in LSTM top-K.')